# sales_line — Integration view

Geïntegreerde business-view op **line-grain**: `INTEGRATION.order_header ⋈ INTEGRATION.order_detail` op `order_id`. Eén rij per `order_detail_id`, met alle header-attributen gedenormaliseerd op iedere regel.

**Waarom een view (geen MV):** geen state nodig, pure join — Spark optimizer pusht filters door naar beide Silver streaming tables. Header-correcties verschijnen direct, geen refresh-lag. Quarantine-rijen zijn per definitie afwezig.

Georkestreerd door `views/07_apply_views.ipynb` (`dbutils.notebook.run`).

In [ ]:
%sql
CREATE WIDGET TEXT catalog DEFAULT "DEMO";
USE CATALOG ${catalog};

CREATE OR REPLACE VIEW INTEGRATION.sales_line AS
SELECT
  -- Line-grain key
  od.order_detail_id,
  od.order_id,
  -- order_detail business columns
  od.menu_item_id,
  od.quantity,
  od.unit_price,
  od.price,
  od.order_item_discount_amount,
  od.line_number,
  -- order_header business columns (denormalised onto each line)
  oh.truck_id,
  oh.location_id,
  oh.customer_id,
  oh.discount_id,
  oh.shift_id,
  oh.shift_start_time,
  oh.shift_end_time,
  oh.order_channel,
  oh.order_ts,
  oh.served_ts,
  oh.order_currency,
  oh.order_amount,
  oh.order_tax_amount,
  oh.order_discount_amount,
  oh.order_total
FROM INTEGRATION.order_detail  od
JOIN INTEGRATION.order_header  oh USING (order_id);